# 🌐 Colab REST API Backend MVP — Qwen3-TTS

Turn Google Colab into a live, publicly-accessible TTS REST API backend using FastAPI + Cloudflare Tunnel.

## Architecture
Client → Cloudflare Tunnel URL → FastAPI (Port 8000) → Qwen3-TTS → Audio Response

## Endpoints
- `GET /` — health check
- `GET /api/speakers` — list available preset speakers
- `POST /api/tts/custom` — generate with preset speaker
- `POST /api/tts/design` — generate with voice description


In [ ]:
# Install dependencies and Cloudflare tunnel executable
!pip install -q qwen-tts soundfile fastapi uvicorn pydantic requests
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

In [ ]:
import uvicorn, fastapi, pydantic, io, base64, threading, subprocess, time
import soundfile as sf, numpy as np, torch
from qwen_tts import Qwen3TTSModel
from IPython.display import Audio, display

In [ ]:
# Load Models
# Using 0.6B for CustomVoice to save VRAM alongside 1.7B for VoiceDesign
print("Loading Custom Voice Model...")
cv_model = Qwen3TTSModel.from_pretrained("Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice", device_map="cuda:0", dtype=torch.bfloat16, attn_implementation="sdpa")

print("Loading Voice Design Model...")
vd_model = Qwen3TTSModel.from_pretrained("Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign", device_map="cuda:0", dtype=torch.bfloat16, attn_implementation="sdpa")

print("\nVRAM Usage (allocated):", torch.cuda.memory_allocated() / 1e9, "GB")

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

app = FastAPI(title="Qwen3-TTS API", version="1.0")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

class CustomTTSRequest(BaseModel):
    text: str
    speaker: str = "Ryan"
    language: str = "English"
    instruct: str = ""

class DesignTTSRequest(BaseModel):
    text: str
    instruct: str
    language: str = "English"

@app.get("/")
def health():
    return {"status": "ok", "model": "Qwen3-TTS"}

@app.get("/api/speakers")
def get_speakers():
    return {"speakers": cv_model.get_supported_speakers()}

@app.post("/api/tts/custom")
def tts_custom(req: CustomTTSRequest):
    wavs, sr = cv_model.generate_custom_voice(text=req.text, language=req.language, speaker=req.speaker, instruct=req.instruct)
    buf = io.BytesIO()
    sf.write(buf, wavs[0], sr, format="WAV")
    buf.seek(0)
    return StreamingResponse(buf, media_type="audio/wav", headers={"Content-Disposition": "attachment; filename=output.wav"})

@app.post("/api/tts/design")
def tts_design(req: DesignTTSRequest):
    wavs, sr = vd_model.generate_voice_design(text=req.text, language=req.language, instruct=req.instruct)
    buf = io.BytesIO()
    sf.write(buf, wavs[0], sr, format="WAV")
    buf.seek(0)
    return StreamingResponse(buf, media_type="audio/wav", headers={"Content-Disposition": "attachment; filename=output.wav"})

In [ ]:
# Start the FastAPI server in a thread and open a Cloudflare tunnel
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)
print("✅ Server running on port 8000")

cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

print("⏳ Starting Cloudflare tunnel...")
PUBLIC_URL = None
while True:
    line = cf_proc.stdout.readline()
    if not line:
        break
    if "trycloudflare.com" in line:
        parts = line.strip().split(" ")
        for p in parts:
            if "trycloudflare.com" in p:
                PUBLIC_URL = p.strip()
                break
        if PUBLIC_URL:
            print(f"\n🌐 YOUR PUBLIC API URL: {PUBLIC_URL}")
            print(f"   Test it: {PUBLIC_URL}/api/speakers")
            break


In [ ]:
import requests
import io

# Test the API endpoints
if PUBLIC_URL:
    print("Testing /...")
    print(requests.get(f"{PUBLIC_URL}/").json())
    
    print("\nTesting /api/speakers...")
    print(requests.get(f"{PUBLIC_URL}/api/speakers").json())
    
    print("\nTesting /api/tts/custom...")
    payload = {"text": "Hello from the API!", "speaker": "Ryan", "language": "English", "instruct": "Happy"}
    res = requests.post(f"{PUBLIC_URL}/api/tts/custom", json=payload)
    if res.status_code == 200:
        display(Audio(res.content))
    else:
        print("Error:", res.status_code)

In [ ]:
html_code = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Qwen3-TTS Web Client</title>
    <style>
        body {{ font-family: sans-serif; padding: 2rem; max-width: 600px; margin: auto; }}
        input, select, textarea {{ width: 100%; padding: 0.5rem; margin-bottom: 1rem; }}
        button {{ padding: 0.5rem 1rem; background: #007bff; color: white; border: none; cursor: pointer; }}
    </style>
</head>
<body>
    <h2>Qwen3-TTS Web Client</h2>
    <textarea id="text" rows="4">Hello world!</textarea>
    <select id="speaker">
        <option value="Ryan">Ryan</option>
        <option value="Bella">Bella</option>
    </select>
    <input type="text" id="instruct" placeholder="Instruction (e.g. happy)">
    <button onclick="generate()">Generate</button>
    <br><br>
    <audio id="audio" controls></audio>

    <script>
        async function generate() {{
            const text = document.getElementById('text').value;
            const speaker = document.getElementById('speaker').value;
            const instruct = document.getElementById('instruct').value;
            
            const res = await fetch("{PUBLIC_URL}/api/tts/custom", {{
                method: 'POST',
                headers: {{ 'Content-Type': 'application/json' }},
                body: JSON.stringify({{ text, speaker, language: "English", instruct }})
            }});
            
            const blob = await res.blob();
            document.getElementById('audio').src = URL.createObjectURL(blob);
        }}
    </script>
</body>
</html>
"""
print("\n\nHTML Client code:\n")
print(html_code)

In [ ]:
# To stop the tunnel
cf_proc.terminate()